# Feature-Based Model

## Objective

This notebook develops a feature-based forecasting model using the XGBoost Regressor.

Temporal and cyclical calendar features are engineered from the timestamp to capture seasonal patterns in weekly German electricity demand. The model is evaluated using RMSE, MAE and MAPE, and its performance is compared with the benchmark, SARIMA and SARIMAX models.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error

from xgboost import XGBRegressor

## Load Dataset

The weekly electricity demand dataset is loaded and prepared for feature engineering.

In [2]:
weekly_load = pd.read_csv(
    "weekly_load.csv",
    index_col=0,
    parse_dates=True
)

weekly_load.head()

,load
timestamp,
2015-01-04 00:00:00+00:00,47233.739583
2015-01-11 00:00:00+00:00,56191.101190
2015-01-18 00:00:00+00:00,57672.678571
2015-01-25 00:00:00+00:00,58613.303571
2015-02-01 00:00:00+00:00,58734.029762


## Forecast Evaluation Function

A reusable evaluation function is defined to calculate RMSE, MAE and MAPE for comparing model performance.

In [3]:
def evaluate_forecast(actual, forecast):

    rmse = np.sqrt(mean_squared_error(actual, forecast))
    mae = mean_absolute_error(actual, forecast)
    mape = np.mean(np.abs((actual - forecast) / actual)) * 100

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape
    }

## Feature Engineering

Calendar-based and cyclical features are generated from the timestamp to capture seasonal patterns in electricity demand.

In [4]:
weekly_load["month"] = weekly_load.index.month

weekly_load["quarter"] = weekly_load.index.quarter

weekly_load["week"] = weekly_load.index.isocalendar().week.astype(int)

weekly_load["year"] = weekly_load.index.year

weekly_load["month_sin"] = np.sin(2*np.pi*weekly_load["month"]/12)

weekly_load["month_cos"] = np.cos(2*np.pi*weekly_load["month"]/12)

weekly_load["week_sin"] = np.sin(2*np.pi*weekly_load["week"]/52)

weekly_load["week_cos"] = np.cos(2*np.pi*weekly_load["week"]/52)

## Train-Test Split

The dataset is divided into training and testing subsets using the final 104 observations as the forecasting horizon.

In [5]:
forecast_horizon = 104

train = weekly_load.iloc[:-forecast_horizon]

test = weekly_load.iloc[-forecast_horizon:]

print("Training observations:", len(train))
print("Testing observations:", len(test))

Training observations: 197
Testing observations: 104
